### This pipeline parses PDFs with grobid.

**input**: last 3 days of records from taxicab where type is pdf from table `openalex.taxicab.taxicab_results`

**process**: grobid API on ECS

**output**: xml from grobid, along with urls and ids into table `openalex.pdf.grobid_processing_results`

#### Result statuses

| status | meaning | retry? |
|---|---|---|
| `success` / `success - cached response` | usable TEI stored | no |
| `success_invalid_xml` | wrapper returned 200 but the TEI is structurally empty | no |
| `failed` + `HTTP error 503/504` | sidecar busy / timed out, or storage write failed | yes — transient |
| `failed` + `HTTP error 422` | grobid cannot process this PDF (pdfalto rejected it, or no content) | no — deterministic |
| `failed` + `HTTP error 413/400` | too large / not a PDF | no — permanent |
| `failed` + `HTTP error 404` | PDF missing from the S3 bucket | only after re-harvest |
| `failed` + `Retries exhausted...` | retry budget spent on a 429/503/504 | yes — transient |

#### Retries and the warm window (oxjob #1698)

- **Normal runs retry transient failures.** A pdf is done once it has a `success*` row or a permanent failure (400/404/413/422). Transient failures (503/504/429, `Retries exhausted`, connection errors, timeouts) are picked up again by the next runs while the pdf is inside the 3-day window, up to `MAX_TRANSIENT_ATTEMPTS` (3) rows. Before this, any prior row excluded a pdf forever.
- **Reparse runs re-queue this run's transient failures** (same 3-in-7-days cap), so a drain that hits a busy or cold GROBID loses no work.
- **GROBID's ECS service is warm only around the daily run** (scheduled scaling on `service/grobid/grobid-service`: 40 tasks from 00:10 UTC, back to 1 at 04:00 UTC; this job runs at 00:30 UTC, before the 05:00 UTC End 2 End reads `PDFWork`). For a drain outside that window, pre-scale first: `aws application-autoscaling register-scalable-target --service-namespace ecs --resource-id service/grobid/grobid-service --scalable-dimension ecs:service:DesiredCount --min-capacity 40 --max-capacity 60`, wait ~10 min for healthy targets, and set `--min-capacity 1` when done.

#### One parse per PDF (oxjob #2023)

- **Reused files.** A harvest whose bytes match a file we already hold returns that file's id instead of storing a copy (`pdf-md5` table). `pdf_works` is keyed by link, so a reused id under a new link gets a copy of the file's latest good parse row for that link (XML read back from `openalex-grobid-xml`, no GROBID call): `link_reused_files()`, normal mode only. It acts only on a fetch newer than the link's latest parse row that names a different file.
- **No double parses.** A reparse run leaves queued PDFs still in the daily run's 3-day window without a finished parse to the daily run, and skips PDFs that got a good parse after they were queued (#1987's 3 Nov drain parsed 1,086,556 PDFs a second time next to the daily run). Both modes parse each PDF id once per run.

#### Reparse Queue Usage

To re-parse records (e.g. after GROBID fixes or to retry the transient classes above), insert into `openalex.pdf.grobid_reparse_queue`, then run the **Parse PDFs** job (`99046008643006`) with `reparse_queue_only=true`. There is no separate reparse job. Queue ids resolve their POST metadata from `taxicab_results` pdf rows, falling back to their own latest success row in `grobid_processing_results` for pre-taxicab-era pdfs (oxjob #789 Stage 4), then to their `pdf_works` row (url + doi/pmh id) for legacy-backfill pdfs that were never in `grobid_processing_results` (oxjob #984). The run dequeues its whole batch, including ids that resolve no metadata from any source.

**Examples:**
```sql
-- Bulk: everything that failed for a transient reason and never later succeeded
INSERT INTO openalex.pdf.grobid_reparse_queue (source_pdf_id)
SELECT DISTINCT source_pdf_id
FROM openalex.pdf.grobid_processing_results
WHERE (error_message LIKE 'HTTP error 503%'
    OR error_message LIKE 'HTTP error 504%'
    OR error_message LIKE 'Retries exhausted%')
  AND source_pdf_id NOT IN (
    SELECT source_pdf_id FROM openalex.pdf.grobid_processing_results WHERE status LIKE 'success%'
  );

-- Single PDF by taxicab_id
INSERT INTO openalex.pdf.grobid_reparse_queue (source_pdf_id)
VALUES ('some-uuid-here');
```

In [ ]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError
import time
import random
from urllib3.util import Retry
from requests.adapters import HTTPAdapter
from datetime import datetime, timedelta
from time import sleep
import re
import requests
from requests.exceptions import Timeout
import json
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, BooleanType, TimestampType

In [0]:
GROBID_URL = "http://grobid-api-load-balancer-1880850154.us-east-1.elb.amazonaws.com/parse"
# oxjob #1782: the load balancers will accept only requests that carry this secret (ALB rules after 2026-10-28).
# Read once on the driver; a missing scope or ACL means no header, so this notebook keeps working either way.
try:
    ORIGIN_HEADERS = {"X-OpenAlex-Origin-Key": dbutils.secrets.get("openalex-harvest", "origin-key")}
except Exception:
    ORIGIN_HEADERS = {}
BATCH_SIZE = 100
MAX_WORKERS = 400

# Must outwait every layer below us: wrapper budget 170s < gunicorn 210s < ALB idle
# 240s. Hanging up early re-sends the request into the same saturated pool while
# the original keeps running to completion server-side, which is the pile-up the
# wrapper's retry exists to stop.
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 270

http_session = requests.Session()

retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    # 503/504 are owned by ONE layer: the manual loop in process_pdf_single,
    # which backs off and records an outcome row. Leaving them here too
    # multiplied the wrapper's 5 attempts by this adapter's 4 and the loop's.
    status_forcelist=[429, 502],
    allowed_methods=["GET", "POST"]
)

adapter = HTTPAdapter(
    pool_connections=MAX_WORKERS,
    pool_maxsize=MAX_WORKERS*2,
    max_retries=retry_strategy
)

http_session.mount("http://", adapter)
http_session.mount("https://", adapter)

In [0]:
grobid_results_schema = StructType([
    StructField("id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("source_pdf_id", StringType(), True),
    StructField("url", StringType(), True),
    StructField("native_id", StringType(), True),
    StructField("native_id_namespace", StringType(), True),
    StructField("s3_key", StringType(), True),
    StructField("s3_path", StringType(), True),
    StructField("xml_content", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("created_date", TimestampType(), True)
])

In [ ]:
def create_error_response(pdf_uuid, url, native_id, native_id_namespace, error_message):
    """Create a standardized error response dictionary"""
    return {
        "id": None,
        "status": "failed",
        "source_pdf_id": pdf_uuid,
        "url": url,
        "native_id": native_id,
        "native_id_namespace": native_id_namespace,
        "s3_key": None,
        "s3_path": None,
        "xml_content": None,
        "error_message": error_message,
        "created_date": datetime.now(),
    }

def process_pdf_single(row_data):
    """Process a single PDF through the GROBID service with improved connection handling"""
    pdf_uuid = row_data.get('source_pdf_id')
    url = row_data.get('url')
    native_id = row_data.get('native_id')
    native_id_namespace = row_data.get('native_id_namespace')
    # the wrapper already makes up to 5 sidecar attempts on its own budget,
    # so this layer only needs one more pass for a genuinely saturated pool
    max_retries = 1
    
    print(f"Processing PDF {pdf_uuid} from {url}")
    start_time = time.time()
    
    data = {
        "pdf_uuid": pdf_uuid,
        "url": url,
        "native_id": native_id,
        "native_id_namespace": native_id_namespace
    }
    
    retry_count = 0
    last_error = None
    while retry_count <= max_retries:
        try:
            if retry_count > 0:
                jitter = random.uniform(0.1, 1.0)
                time.sleep(jitter)
                
            response = http_session.post(
                GROBID_URL,
                json=data,
                headers=ORIGIN_HEADERS,
                timeout=(CONNECT_TIMEOUT, READ_TIMEOUT)
            )
            
            if response.status_code in [429, 503, 504]:
                retry_count += 1
                last_error = f"HTTP error {response.status_code}: {response.text[:200]}"
                # exponential backoff with jitter
                wait_time = min(2 ** retry_count + random.uniform(0, 1), 60)
                print(f"Service error {response.status_code} for {url}, retrying in {wait_time:.2f} seconds... ({retry_count}/{max_retries})")
                time.sleep(wait_time)
                continue

            if response.status_code >= 400:
                error_message = f"HTTP error: {response.status_code}"
                try:
                    response_json = response.json()
                    if "error" in response_json and response_json["error"] is not None:
                        error_message = f"HTTP error {response.status_code}: {response_json['error']}"
                    else:
                        error_message = f"HTTP error {response.status_code}: {response.text[:200]}"
                except:
                    # If not valid JSON, use text
                    error_message = f"HTTP error {response.status_code}: {response.text[:200]}"
                
                return create_error_response(
                    pdf_uuid, url, native_id, native_id_namespace,
                    error_message
                )

            response.raise_for_status()
            
            result = response.json()
            
            end_time = time.time()
            processing_time = end_time - start_time
            print(f"GROBID request for {pdf_uuid} took {processing_time:.2f} seconds")
            
            # successful response
            return {
                "id": result.get("id"),
                "status": result.get("status") or "success",
                "source_pdf_id": result.get("source_pdf_id"),
                "url": url,
                "native_id": native_id,
                "native_id_namespace": native_id_namespace,
                "s3_key": result.get("s3_key"),
                "s3_path": result.get("s3_path"),
                "xml_content": result.get("xml_content"),
                "error_message": None,
                "created_date": datetime.now(),
            }
                
        except requests.exceptions.ConnectionError as e:
            retry_count += 1
            error_msg = f"Connection error: {str(e)}"
            last_error = error_msg
            
            # If exceeded retries, return error
            if retry_count > max_retries:
                print(f"Failed to connect for {url} after {max_retries} retries: {error_msg}")
                return create_error_response(
                    pdf_uuid, url, native_id, native_id_namespace,
                    error_msg
                )
            
            # longer wait for connection errors
            wait_time = min(5 ** retry_count + random.uniform(0, 2), 120)
            print(f"Connection error for {url}, retrying in {wait_time:.2f} seconds... ({retry_count}/{max_retries})")
            time.sleep(wait_time)
            
        except Exception as e:
            retry_count += 1
            error_msg = str(e)
            last_error = error_msg
            
            # if exceeded retries, return error
            if retry_count > max_retries:
                print(f"Error processing {url}: {error_msg}")
                return create_error_response(
                    pdf_uuid, url, native_id, native_id_namespace,
                    error_msg
                )
            
            # wait before retrying with jitter
            wait_time = min(2 ** retry_count + random.uniform(0, 1), 60)
            print(f"Error: {error_msg}, retrying in {wait_time:.2f} seconds... ({retry_count}/{max_retries})")
            time.sleep(wait_time)

    # Retry budget spent on a retryable status (429/503/504). Without this the
    # function fell through returning None, which recorded no row at all and
    # poisoned the pandas -> spark write for the whole accumulated batch.
    print(f"Retries exhausted for {url}: {last_error}")
    return create_error_response(
        pdf_uuid, url, native_id, native_id_namespace,
        f"Retries exhausted after {max_retries} attempts: {last_error}"
    )

In [0]:
def process_pdfs_with_continuous_batching(candidates_df, write_size=BATCH_SIZE*10):
    """Process PDFs with continuous batching that doesn't wait for stragglers"""
    
    # Convert to pandas for processing
    pdf_data = candidates_df.toPandas()
    
    # Create a list of row data dictionaries
    rows = []
    for _, row in pdf_data.iterrows():
        rows.append({
            'source_pdf_id': row['source_pdf_id'],
            'url': row['url'],
            'native_id': row['native_id'],
            'native_id_namespace': row['native_id_namespace']
        })
    
    print(f"Processing {len(rows)} records with continuous batching")
    print(f"Will write to database in batches of {write_size} records")
    
    # Create a bounded thread pool
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks to the executor without waiting
        future_to_row = {executor.submit(process_pdf_single, row): row for row in rows}
        
        # Process results as they complete rather than waiting for all
        completed_count = 0
        accumulated_results = []
        
        for future in as_completed(list(future_to_row)):
            # release the completed future immediately: retained futures keep their
            # result (the full TEI) alive, and a 1M-row batch OOMs the driver
            row = future_to_row.pop(future)
            try:
                result = future.result(timeout=READ_TIMEOUT * 2 + 60)  # must clear a full client retry chain
                accumulated_results.append(result)
                completed_count += 1
                
                if completed_count % 50 == 0:
                    print(f"Completed {completed_count}/{len(rows)} records")
                
                # Write in larger chunks
                if len(accumulated_results) >= write_size:
                    print(f"Writing batch of {len(accumulated_results)} results to database")
                    result_df = pd.DataFrame(accumulated_results)
                    spark_df = spark.createDataFrame(result_df, schema=grobid_results_schema)
                    write_results_to_table(spark_df)
                    accumulated_results = []
                    
            except TimeoutError:
                print(f"Task timed out for PDF: {row.get('url')}")
                accumulated_results.append({
                    "id": None,
                    "status": "timeout",
                    "source_pdf_id": row.get('source_pdf_id'),
                    "url": row.get('url'),
                    "native_id": row.get('native_id'),
                    "native_id_namespace": row.get('native_id_namespace'),
                    "s3_key": None,
                    "s3_path": None,
                    "xml_content": None,
                    "error_message": "Task timed out",
                    "created_date": datetime.now(),
                })
            except Exception as e:
                print(f"Error processing PDF {row.get('url')}: {str(e)}")
                accumulated_results.append({
                    "id": None,
                    "status": "error",
                    "source_pdf_id": row.get('source_pdf_id'),
                    "url": row.get('url'),
                    "native_id": row.get('native_id'),
                    "native_id_namespace": row.get('native_id_namespace'),
                    "s3_key": None,
                    "s3_path": None,
                    "xml_content": None,
                    "error_message": str(e),
                    "created_date": datetime.now(),
                })
        
        # Write any remaining results
        if accumulated_results:
            print(f"Writing final batch of {len(accumulated_results)} results to database")
            result_df = pd.DataFrame(accumulated_results)
            spark_df = spark.createDataFrame(result_df, schema=grobid_results_schema)
            write_results_to_table(spark_df)
    
    print(f"Successfully processed {completed_count} records")
    return completed_count

In [ ]:
RUN_STATUS_COUNTS = Counter()

def write_results_to_table(results_df):
    """Write results to the grobid_processing_results table"""
    if results_df is None or results_df.isEmpty():
        print("No results to write")
        return

    # create the table if it doesn't exist
    spark.sql("""
    CREATE TABLE IF NOT EXISTS openalex.pdf.grobid_processing_results (
        id STRING,
        status STRING,
        source_pdf_id STRING,
        url STRING,
        native_id STRING,
        native_id_namespace STRING,
        s3_key STRING,
        s3_path STRING,
        xml_content STRING,
        error_message STRING,
        created_date TIMESTAMP
    )
    USING DELTA
    """)

    # Mark structurally-bad GROBID output (empty <body/> TEI + error-wrapper
    # markers + non-TEI bodies) so it doesn't propagate downstream. Keeps the
    # row + source_pdf_id for dedupe so we don't re-call GROBID next cycle on
    # the same content. Predicates mirror notebooks/scraping/classify_grobid_xmls.py
    # (oxjob #202). Companion filter sits in notebooks/ingest/PDF.py grobid_raw.
    is_bad = (
        F.col("status").startswith("success")
        & (
            F.col("xml_content").isNull()
            | F.col("xml_content").like("%[BAD_INPUT_DATA]%")
            | F.col("xml_content").like("%[NO_BLOCKS]%")
            | F.col("xml_content").like("%[TIMEOUT]%")
            | F.col("xml_content").like("%[NO_GROBID_RESPONSES]%")
            | F.col("xml_content").like("%<body/>%")
            | ~F.lower(F.substring(F.col("xml_content"), 1, 200)).like("%<tei%")
        )
    )
    results_df = (
        results_df
        .withColumn("_bad", is_bad)
        .withColumn(
            "error_message",
            F.when(F.col("_bad"), F.lit("invalid_xml_content"))
             .otherwise(F.col("error_message")),
        )
        .withColumn(
            "status",
            F.when(F.col("_bad"), F.lit("success_invalid_xml"))
             .otherwise(F.col("status")),
        )
        .withColumn(
            "xml_content",
            F.when(F.col("_bad"), F.lit(None).cast("string"))
             .otherwise(F.col("xml_content")),
        )
        .drop("_bad")
    )

    results_df.write.format("delta").mode("append").saveAsTable("openalex.pdf.grobid_processing_results")

    # Tally post-relabel statuses so the run can report its own failure rate.
    batch_counts = {r["status"]: r["count"] for r in results_df.groupBy("status").count().collect()}
    RUN_STATUS_COUNTS.update(batch_counts)

    print(f"Successfully appended {sum(batch_counts.values())} records to grobid_processing_results")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS openalex.pdf.grobid_reparse_queue (
    source_pdf_id STRING,
    created_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
""")

In [ ]:
RUN_STARTED_AT = datetime.now()
MAX_TRANSIENT_ATTEMPTS = 3
# A row that ends a pdf's parsing: usable or invalid TEI, or a permanent failure
# (400 not a PDF, 404 file missing, 413 too large, 422 unparseable). Everything else
# (503/504/429, "Retries exhausted", connection errors, timeouts) is transient.
IS_TERMINAL = (
    F.col("status").startswith("success")
    | F.coalesce(F.col("error_message"), F.lit("")).rlike(r"^HTTP error:? ?(400|404|413|422)\b")
)

dbutils.widgets.text("reparse_queue_only", "false", "Reparse queue only (true/false)")
dbutils.widgets.text("batch_size", "500000", "Max records to process per run (reparse mode only)")
reparse_queue_only = dbutils.widgets.get("reparse_queue_only").strip().lower() == "true"
batch_size = int(dbutils.widgets.get("batch_size"))

if reparse_queue_only:
    print(f"REPARSE MODE: processing up to {batch_size:,} records from queue")
    queue_df = spark.read.table("openalex.pdf.grobid_reparse_queue")
    queue_count = queue_df.count()
    if queue_count == 0:
        dbutils.notebook.exit("Queue empty — nothing to reparse")
    print(f"Reparse queue has {queue_count:,} records")

    # Take a batch from the queue. Cached because limit() has no ordering and is
    # not stable across re-evaluation — the dequeue step reuses these exact rows.
    queue_batch = queue_df.limit(batch_size).cache()
    queue_batch_count = queue_batch.count()

    # One parse per pdf (oxjob #2023): leave pdfs the daily run will take (fetched in the last 2 days, so still inside
    # its 3-day window at its next start, and no finished parse) to it, and skip pdfs that got a good parse after they
    # were queued. The whole batch is still dequeued.
    queued = queue_batch.groupBy("source_pdf_id").agg(F.min("created_date").alias("queued_at"))
    queued_rows = spark.read.table("openalex.pdf.grobid_processing_results").join(queued.select("source_pdf_id"), "source_pdf_id", "left_semi")
    daily_run_takes = (
        spark.read.table("openalex.taxicab.taxicab_results")
        .filter(F.col("taxicab_id").isNotNull() & F.col("content_type").contains("pdf")
                & (F.col("processed_date") >= F.lit(RUN_STARTED_AT - timedelta(days=2))))
        .select(F.col("taxicab_id").alias("source_pdf_id"))
        .join(queued.select("source_pdf_id"), "source_pdf_id", "left_semi")
        .join(queued_rows.filter(IS_TERMINAL).select("source_pdf_id"), "source_pdf_id", "left_anti")
    )
    parsed_since_queued = (
        queued_rows.filter(F.col("status").startswith("success"))
        .join(queued, "source_pdf_id")
        .filter(F.col("created_date") > F.coalesce(F.col("queued_at"), F.lit(RUN_STARTED_AT)))
        .select("source_pdf_id")
    )
    to_parse = queued.select("source_pdf_id").join(daily_run_takes.union(parsed_since_queued), "source_pdf_id", "left_anti").cache()
    skipped_count = queue_batch_count - to_parse.count()
    print(f"{skipped_count:,} queued rows skipped: duplicates, left to the daily run, or parsed since queued")

    taxicab_meta = (
        spark.read.table("openalex.taxicab.taxicab_results")
        .filter(
            F.col("taxicab_id").isNotNull() &
            F.col("content_type").contains("pdf")
        )
        .select(
            F.col("taxicab_id").alias("source_pdf_id"),
            F.col("url").alias("tx_url"),
            F.col("native_id").alias("tx_native_id"),
            F.col("native_id_namespace").alias("tx_native_id_namespace"),
        )
    )

    # Pre-taxicab-era pdfs (oxjob #789 Stage 4) have no taxicab row at all, but
    # the wrapper fetches the file from R2 by uuid, so taxicab absence is not a
    # parse blocker — fall back to the metadata their own latest success row
    # recorded. Left-semi first keeps the aggregation queue-sized.
    gpr_meta = (
        spark.read.table("openalex.pdf.grobid_processing_results")
        .join(to_parse, "source_pdf_id", "left_semi")
        .filter(F.col("status").startswith("success"))
        .groupBy("source_pdf_id")
        .agg(
            F.max_by("url", "created_date").alias("gpr_url"),
            F.max_by("native_id", "created_date").alias("gpr_native_id"),
            F.max_by("native_id_namespace", "created_date").alias("gpr_native_id_namespace"),
        )
    )

    # Legacy-backfill pdfs (oxjob #984) predate grobid_processing_results
    # entirely — no taxicab row, no prior success row — but their pdf_works
    # row still carries the url (which is also the pdf_works key, so the fresh
    # parse replaces the legacy row) and the work's own id in `ids`.
    pw_meta = (
        spark.read.table("openalex.pdf.pdf_works")
        .select(
            F.expr("regexp_replace(get(filter(ids, x -> x.namespace = 'docs.pdf'), 0).id, '\\\\.pdf$', '')").alias("source_pdf_id"),
            F.col("native_id").alias("pw_url"),
            F.expr("coalesce(get(filter(ids, x -> x.namespace = 'doi'), 0).id, get(filter(ids, x -> x.namespace = 'pmh'), 0).id, native_id)").alias("pw_native_id"),
            F.expr("CASE WHEN exists(ids, x -> x.namespace = 'doi') THEN 'doi' WHEN exists(ids, x -> x.namespace = 'pmh') THEN 'pmh' ELSE 'url' END").alias("pw_native_id_namespace"),
        )
        .filter(F.col("source_pdf_id").isNotNull())
        .join(to_parse, "source_pdf_id", "left_semi")
        .groupBy("source_pdf_id")
        .agg(
            F.max("pw_url").alias("pw_url"),
            F.max("pw_native_id").alias("pw_native_id"),
            F.max("pw_native_id_namespace").alias("pw_native_id_namespace"),
        )
    )

    candidates_df = (
        to_parse
        .join(taxicab_meta, "source_pdf_id", "left")
        .join(gpr_meta, "source_pdf_id", "left")
        .join(pw_meta, "source_pdf_id", "left")
        .select(
            "source_pdf_id",
            F.coalesce("tx_url", "gpr_url", "pw_url").alias("url"),
            F.coalesce("tx_native_id", "gpr_native_id", "pw_native_id").alias("native_id"),
            F.coalesce("tx_native_id_namespace", "gpr_native_id_namespace", "pw_native_id_namespace").alias("native_id_namespace"),
        )
        .filter(F.col("url").isNotNull())
    )
else:
    print("NORMAL MODE: finding new records to parse")
    three_days_ago = datetime.now() - timedelta(days=3)

    taxicab_df = spark.table("openalex.taxicab.taxicab_results") \
        .filter(
            (F.col("taxicab_id").isNotNull()) & 
            (F.col("content_type").contains("pdf")) &
            (F.col("processed_date") >= F.lit(three_days_ago))
        )

    # Skip only pdfs that are done (a terminal row) or have used their transient retries.
    # Aggregating just the window's ids keeps this as cheap as the old distinct() scan.
    try:
        window_ids = taxicab_df.select(F.col("taxicab_id").alias("source_pdf_id")).distinct()
        existing_ids = (
            spark.table("openalex.pdf.grobid_processing_results")
            .join(window_ids, "source_pdf_id", "left_semi")
            .groupBy("source_pdf_id")
            .agg(
                F.max(IS_TERMINAL.cast("int")).alias("terminal"),
                F.count(F.lit(1)).alias("attempts"),
            )
            .filter((F.col("terminal") == 1) | (F.col("attempts") >= MAX_TRANSIENT_ATTEMPTS))
            .select("source_pdf_id")
        )
    except:
        print("No existing grobid_processing_results table found. Creating new.")
        existing_ids = spark.createDataFrame([], schema=StructType([StructField("source_pdf_id", StringType(), True)]))

    candidates_df = taxicab_df \
        .join(existing_ids, taxicab_df["taxicab_id"] == existing_ids["source_pdf_id"], "left_anti") \
        .select(
            F.col("taxicab_id").alias("source_pdf_id"),
            "url",
            "native_id",
            "native_id_namespace"
        ) \
        .dropDuplicates(["source_pdf_id"])  # a reused file can arrive under several links; link_reused_files() covers the rest

candidates_df = candidates_df.cache()
total_records = candidates_df.count()
print(f"Found {total_records:,} records to process")

In [ ]:
def print_run_summary():
    """Report this run's own outcome mix, so a regression in the GROBID failure
    rate is visible in the run log instead of only surfacing downstream."""
    total = sum(RUN_STATUS_COUNTS.values())
    if not total:
        return
    bad_xml = RUN_STATUS_COUNTS.get("success_invalid_xml", 0)
    usable = sum(n for s, n in RUN_STATUS_COUNTS.items() if s.startswith("success")) - bad_xml
    failed = total - usable - bad_xml
    print(
        f"GROBID run summary: {total:,} written | "
        f"{usable:,} usable ({100.0 * usable / total:.2f}%) | "
        f"{bad_xml:,} invalid xml ({100.0 * bad_xml / total:.2f}%) | "
        f"{failed:,} failed ({100.0 * failed / total:.2f}%)"
    )
    for status, n in RUN_STATUS_COUNTS.most_common():
        print(f"  {status}: {n:,}")


def dequeue_batch():
    """Drop the whole pulled batch from the queue — including ids that yielded no
    candidate row. Those are unprocessable, and dequeuing only the processed rows
    left them to be re-selected by every subsequent run."""
    unprocessable = queue_batch_count - skipped_count - total_records
    if unprocessable > 0:
        print(f"WARNING: {unprocessable:,} queued ids had no url from taxicab_results, prior parse rows, or pdf_works — dequeued unprocessed")

    queue_table = spark.read.table("openalex.pdf.grobid_reparse_queue")
    processed_ids = queue_batch.select("source_pdf_id")
    remaining = queue_table.join(
        processed_ids,
        queue_table["source_pdf_id"] == processed_ids["source_pdf_id"],
        "left_anti"
    ).cache()
    remaining_count = remaining.count()  # materialize before overwriting the source table
    remaining.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("openalex.pdf.grobid_reparse_queue")
    remaining.unpersist()
    print(f"Removed {queue_batch_count:,} from queue. {remaining_count:,} remaining.")


def requeue_transient_failures():
    """Put back the ids whose parse failed transiently in this run (busy or cold GROBID), so a
    drain that ran while the service was scaling loses no work. An id with MAX_TRANSIENT_ATTEMPTS
    transient failures in 7 days stays out; the bulk re-queue SQL above still finds it (oxjob #1698)."""
    recent = (
        spark.read.table("openalex.pdf.grobid_processing_results")
        .join(queue_batch.select("source_pdf_id"), "source_pdf_id", "left_semi")
        .filter(F.col("created_date") >= F.lit(RUN_STARTED_AT - timedelta(days=7)))
    )
    per_id = recent.groupBy("source_pdf_id").agg(
        F.max_by(IS_TERMINAL.cast("int"), "created_date").alias("last_terminal"),
        F.max("created_date").alias("last_at"),
        F.sum((~IS_TERMINAL).cast("int")).alias("transient_7d"),
    )
    retry_ids = per_id.filter(
        (F.col("last_terminal") == 0)
        & (F.col("last_at") >= F.lit(RUN_STARTED_AT))
        & (F.col("transient_7d") < MAX_TRANSIENT_ATTEMPTS)
    ).select("source_pdf_id").cache()
    n = retry_ids.count()
    if n:
        (retry_ids.withColumn("created_date", F.current_timestamp())
            .write.format("delta").mode("append").saveAsTable("openalex.pdf.grobid_reparse_queue"))
    retry_ids.unpersist()
    print(f"Re-queued {n:,} transient failures from this run")


XML_BUCKET = "openalex-grobid-xml"
COPY_CHUNK = 20000


def link_reused_files():
    """A harvest can return a file we already hold (same bytes, oxjob #2023), so a pdf id can arrive under a new link.
    pdf_works is keyed by link: give that link a copy of the file's latest good parse row, its XML read back from R2
    (the row's own copy would mean scanning all of grobid_processing_results). Only for a fetch newer than the link's
    latest parse row that names a different file; nothing to do while every fetch stores a new file."""
    import gzip
    import boto3
    window = (
        spark.table("openalex.taxicab.taxicab_results")
        .filter(F.col("taxicab_id").isNotNull() & F.col("content_type").contains("pdf")
                & (F.col("processed_date") >= F.lit(RUN_STARTED_AT - timedelta(days=3))))
        .groupBy(F.col("taxicab_id").alias("source_pdf_id"), "url")
        .agg(F.max_by("native_id", "processed_date").alias("native_id"),
             F.max_by("native_id_namespace", "processed_date").alias("native_id_namespace"),
             F.max("processed_date").alias("fetched_at"))
    )
    good = (spark.table("openalex.pdf.grobid_processing_results").filter(F.col("status").startswith("success"))
            .select("id", "status", "source_pdf_id", "url", "s3_key", "s3_path", "error_message", "created_date"))
    file_parse = (
        good.join(window.select("source_pdf_id").distinct(), "source_pdf_id", "left_semi")
        .groupBy("source_pdf_id")
        .agg(F.max_by(F.struct("id", "status", "s3_key", "s3_path", "error_message"), F.struct("created_date", "id")).alias("p"))
        .select("source_pdf_id", "p.*")
    )
    link_latest = (
        good.join(window.select("url").distinct(), "url", "left_semi")
        .groupBy("url")
        .agg(F.max_by("source_pdf_id", F.struct("created_date", "id")).alias("link_pdf_id"), F.max("created_date").alias("link_parsed_at"))
    )
    copies = (
        window.join(file_parse, "source_pdf_id")
        .join(link_latest, "url", "left")
        .filter(F.col("link_pdf_id").isNull()
                | ((F.col("link_pdf_id") != F.col("source_pdf_id")) & (F.col("fetched_at") > F.col("link_parsed_at"))))
        .select("id", "status", "source_pdf_id", "url", "native_id", "native_id_namespace", "s3_key", "s3_path", "error_message")
        .toPandas()
    )
    if copies.empty:
        print("Linked 0 reused files to new links")
        return
    r2 = boto3.client(
        "s3", region_name="auto",
        endpoint_url=dbutils.secrets.get("cloudflare", "r2-endpoint"),
        aws_access_key_id=dbutils.secrets.get("cloudflare", "r2-access-key-id"),
        aws_secret_access_key=dbutils.secrets.get("cloudflare", "r2-secret-access-key"),
    )

    def read_xml(row):
        # success_invalid_xml rows carry no XML (write_results_to_table nulls it); a file that is gone makes a shell row.
        # Any other error leaves the link out of this run so the next run retries it.
        if row["status"] == "success_invalid_xml" or not row["s3_key"]:
            return None
        try:
            return gzip.decompress(r2.get_object(Bucket=XML_BUCKET, Key=row["s3_key"])["Body"].read()).decode("utf-8")
        except r2.exceptions.NoSuchKey:
            return None
        except Exception as e:
            print(f"XML read failed for {row['s3_key']}, retried next run: {e}")
            return RETRY

    RETRY = object()
    rows = copies.to_dict("records")
    missing = retried = 0
    for start in range(0, len(rows), COPY_CHUNK):
        chunk = rows[start:start + COPY_CHUNK]
        with ThreadPoolExecutor(max_workers=64) as ex:
            xmls = list(ex.map(read_xml, chunk))
        now = datetime.now()
        out = []
        for row, x in zip(chunk, xmls):
            if x is RETRY:
                retried += 1
                continue
            missing += x is None and row["status"] != "success_invalid_xml"
            out.append({**row, "xml_content": x, "created_date": now})
        if out:
            spark.createDataFrame(out, grobid_results_schema).write.format("delta").mode("append").saveAsTable(
                "openalex.pdf.grobid_processing_results")
    print(f"Linked {len(rows) - retried:,} reused files to new links (copied parse rows, no GROBID call; "
          f"{missing:,} without an XML file; {retried:,} left for the next run)")


if total_records > 0:
    write_batch_size = BATCH_SIZE * 10
    
    print(f"Using continuous batching method with {MAX_WORKERS} workers")
    print(f"Will write to database in batches of {write_batch_size} records")
    
    processed_count = process_pdfs_with_continuous_batching(
        candidates_df, 
        write_size=write_batch_size
    )
    
    print(f"Successfully processed {processed_count} of {total_records} records")
    print_run_summary()

    if reparse_queue_only:
        dequeue_batch()
        requeue_transient_failures()

    candidates_df.unpersist()
    print("GROBID PDF processing job completed successfully")
else:
    # Still dequeue in reparse mode: a batch where nothing joined would otherwise
    # be re-pulled forever and block the rest of the queue from draining.
    if reparse_queue_only:
        dequeue_batch()
    print("No records to process. Exiting.")

if reparse_queue_only:
    to_parse.unpersist()
    queue_batch.unpersist()
else:
    link_reused_files()